<a href="https://colab.research.google.com/github/arxsyy/MachineLearning_2026/blob/main/JS09_Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalasi library yang dibutuhkan untuk algoritma pencarian ANN
!pip install faiss-cpu annoy hnswlib pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 8.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 27.7 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520032 sha256=4f2cdf6bd6917ffbf0c8930b01040d3806dd9210404c338ae1b7000025a95090
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=cd36a4b2164d38d7fc281d48c12881405ee8da3f5f684df69c4c24cb80108f2a
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [2]:
# Import library yang akan digunakan
import numpy as np
import time
import pandas as pd
import faiss
from annoy import AnnoyIndex
import hnswlib

Persiapan Dataset

In [3]:
# Pembuatan dataset dummy untuk 1000 data 2D dan 1.000.000 data 5D
np.random.seed(42) # Agar data konsisten saat diulang

# Dataset 1: 1000 data, 2 Dimensi
data_1k_2d = np.random.random((1000, 2)).astype('float32')

# Dataset 2: 1.000.000 data, 5 Dimensi
data_1m_5d = np.random.random((1000000, 5)).astype('float32')

# Data query (data yang akan dicari tetangganya)
query_2d = np.random.random((1, 2)).astype('float32')
query_5d = np.random.random((1, 5)).astype('float32')

Implementasi & Pengukuran ANNOY

In [4]:
# Fungsi untuk membangun index dan mencari jarak menggunakan ANNOY
def test_annoy(data, query, metric):
    dim = data.shape[1]

    # Membangun Index
    t_start_build = time.time()
    index = AnnoyIndex(dim, metric) # metric bisa 'angular' atau 'euclidean'
    for i, vec in enumerate(data):
        index.add_item(i, vec)
    index.build(10) # 10 trees
    t_build = time.time() - t_start_build

    # Pencarian
    t_start_search = time.time()
    index.get_nns_by_vector(query[0], 5) # cari 5 tetangga terdekat
    t_search = time.time() - t_start_search

    return t_build + t_search # Total waktu

Implementasi & Pengukuran FAISS

In [5]:
# Fungsi untuk membangun index dan mencari jarak menggunakan FAISS
def test_faiss(data, query, metric_type):
    dim = data.shape[1]

    # Memilih jenis metrik jarak (L2/Euclidean atau Inner Product)
    t_start = time.time()
    if metric_type == 'euclidean':
        index = faiss.IndexFlatL2(dim)
    else:
        # Untuk Cosine similarity biasanya menggunakan Inner Product pada vektor yang dinormalisasi
        faiss.normalize_L2(data)
        faiss.normalize_L2(query)
        index = faiss.IndexFlatIP(dim)

    index.add(data)

    # Pencarian
    D, I = index.search(query, 5)
    t_total = time.time() - t_start

    return t_total

Implementasi & Pengukuran HNSW

In [6]:
# Fungsi untuk membangun index dan mencari jarak menggunakan HNSW
def test_hnsw(data, query, metric_type):
    dim = data.shape[1]
    num_elements = data.shape[0]

    t_start = time.time()
    # Inisialisasi index (l2 atau cosine)
    index = hnswlib.Index(space=metric_type, dim=dim)
    index.init_index(max_elements=num_elements, ef_construction=100, M=16)

    # Menambahkan data ke index
    index.add_items(data)

    # Pencarian
    index.set_ef(50)
    labels, distances = index.knn_query(query, k=5)
    t_total = time.time() - t_start

    return t_total

Eksekusi Skenario Percobaan

In [7]:
# Eksekusi percobaan untuk semua skenario (2 dimensi data & 2 metric)
# Menyimpan hasil waktu eksekusi dalam list dictionary
hasil_eksperimen = []

skenario = [
    {"nama": "1000/2D (Euclidean)", "data": data_1k_2d, "query": query_2d, "ann_metric": "euclidean", "faiss_metric": "euclidean", "hnsw_metric": "l2"},
    {"nama": "1000/2D (Cosine/Angular)", "data": data_1k_2d.copy(), "query": query_2d.copy(), "ann_metric": "angular", "faiss_metric": "inner_product", "hnsw_metric": "cosine"},
    {"nama": "1000000/5D (Euclidean)", "data": data_1m_5d, "query": query_5d, "ann_metric": "euclidean", "faiss_metric": "euclidean", "hnsw_metric": "l2"},
    {"nama": "1000000/5D (Cosine/Angular)", "data": data_1m_5d.copy(), "query": query_5d.copy(), "ann_metric": "angular", "faiss_metric": "inner_product", "hnsw_metric": "cosine"}
]

for s in skenario:
    waktu_annoy = test_annoy(s["data"], s["query"], s["ann_metric"])
    waktu_faiss = test_faiss(s["data"], s["query"], s["faiss_metric"])
    waktu_hnsw = test_hnsw(s["data"], s["query"], s["hnsw_metric"])

    hasil_eksperimen.append({
        "Jumlah Data/Dimensi (Metric)": s["nama"],
        "ANNOY (detik)": f"{waktu_annoy:.5f}",
        "FAISS (detik)": f"{waktu_faiss:.5f}",
        "HNSW (detik)": f"{waktu_hnsw:.5f}"
    })

Menampilkan Hasil dalam Tabel

In [8]:
# Rekapitulasi hasil uji performa dalam bentuk tabel DataFrame
df_hasil = pd.DataFrame(hasil_eksperimen)
df_hasil

,Jumlah Data/Dimensi (Metric),ANNOY (detik),FAISS (detik),HNSW (detik)
0,1000/2D (Euclidean),0.04333,0.00030,0.06267
1,1000/2D (Cosine/Angular),0.09040,0.00025,0.09418
2,1000000/5D (Euclidean),25.39183,0.02285,89.84237
3,1000000/5D (Cosine/Angular),28.19778,0.02216,83.74433
